In [ ]:
# =============================================================================
# 📦 CELL 1: GPU 확인
# =============================================================================
!nvidia-smi

In [ ]:
# =============================================================================
# 📦 CELL 2: 시스템 설정
# =============================================================================
!apt-get update && apt-get install -y zstd pciutils

In [ ]:
# =============================================================================
# 📦 CELL 3: Ollama 설치
# =============================================================================
!curl -fsSL https://ollama.com/install.sh | sh

In [ ]:
# =============================================================================
# 📦 CELL 4: Ollama 서버 시작 (A100 최적화)
# =============================================================================
import subprocess, time, os

# A100 최적화 설정
os.environ["OLLAMA_FLASH_ATTENTION"] = "1"
os.environ["OLLAMA_NUM_PARALLEL"] = "100"  # 배치 사이즈에 맞춤

subprocess.Popen(["ollama", "serve"])
time.sleep(5)
print("✅ Ollama 서버 시작됨")

In [ ]:
# =============================================================================
# 📦 CELL 5: 모델 다운로드
# =============================================================================
!ollama pull cookieshake/a.x-4.0-light-imatrix:Q8_0

In [ ]:
# =============================================================================
# 📦 CELL 6: 패키지 설치 & Drive 마운트
# =============================================================================
!pip install -q langchain langchain-ollama langchain-core tqdm

from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# =============================================================================
# 📦 CELL 7: 임포트 & 경로 설정
# =============================================================================
import json
import re
import glob
import os
import sys
from tqdm import tqdm
from typing import Any

from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import load_prompt
from langchain_ollama import ChatOllama

# 경로 설정
DRIVE_BASE = "/content/drive/MyDrive/tzudong/contextual-retrieval"
DATA_PATH = f"{DRIVE_BASE}/data"
PROMPTS_PATH = f"{DRIVE_BASE}/prompts"
CHECKPOINT_PATH = f"{DATA_PATH}/checkpoints"
OUTPUT_PATH = f"{DATA_PATH}/output"

sys.path.insert(0, f'{DRIVE_BASE}/src/utils')
from chunk_utils import create_chunks_with_overlap

os.makedirs(CHECKPOINT_PATH, exist_ok=True)
os.makedirs(OUTPUT_PATH, exist_ok=True)

print("✅ 경로 설정 완료")

In [ ]:
# =============================================================================
# 📦 CELL 8: 모델 설정 (A100 최적화)
# =============================================================================
MODEL_NAME = "cookieshake/a.x-4.0-light-imatrix:Q8_0"
BATCH_SIZE = 100  # ⚡ A100 40GB 기준, 더 늘려도 됨 (150까지 테스트해볼 것)
MAX_RETRIES = 1
MAX_CHARS = 300

# 프롬프트 로드
prompt = load_prompt(f"{PROMPTS_PATH}/generate_context_en.yaml", encoding="utf-8")
parse_error_prompt = load_prompt(f"{PROMPTS_PATH}/parse_error_context.yaml", encoding="utf-8")

# LLM 초기화
llm = ChatOllama(model=MODEL_NAME, temperature=0)

print(f"✅ 모델 설정 완료: {MODEL_NAME}")
print(f"📦 배치 사이즈: {BATCH_SIZE}")

In [ ]:
# =============================================================================
# 📦 CELL 9: 유틸리티 함수
# =============================================================================
def read_jsonl(data_path: str) -> dict | None:
    """JSONL 파일에서 마지막 줄 읽기"""
    try:
        with open(data_path, "r", encoding="utf-8") as f:
            lines = f.readlines()
            if lines:
                return json.loads(lines[-1])
    except Exception as e:
        pass
    return None


def get_matching_metadata(meta_path: str, recollect_id: int) -> dict | None:
    """메타데이터에서 recollect_id 매칭"""
    try:
        with open(meta_path, "r", encoding="utf-8") as f:
            for line in reversed(f.readlines()):
                if line.strip():
                    meta = json.loads(line)
                    if meta.get("recollect_id") == recollect_id:
                        return meta
    except:
        pass
    return None


def is_valid_context(text: str) -> bool:
    """마크다운 형식 검사"""
    invalid_patterns = [
        r"^\s*[-*•]\s",
        r"\*\*.*?\*\*",
        r"^#",
        r":\s*$",
        r"^\d+\.\s",
    ]
    for pattern in invalid_patterns:
        if re.search(pattern, text, re.MULTILINE):
            return False
    return True

print("✅ 유틸리티 함수 정의 완료")

In [ ]:
# =============================================================================
# 📦 CELL 10: 배치 처리 함수 (⚡ 핵심 최적화)
# =============================================================================
def batch_generate_contexts_optimized(items: list[dict]) -> list[str]:
    """
    대용량 배치 처리 - A100 최적화
    - 모든 청크를 한번에 처리
    - 실패한 것들도 배치로 재처리
    """
    chain = prompt | llm | StrOutputParser()
    parse_chain = parse_error_prompt | llm | StrOutputParser()
    
    # 입력 구성
    inputs = [
        {
            "title": item["title"],
            "full_transcript": item["full_transcript"],
            "chunk": item["chunk"]
        }
        for item in items
    ]
    
    # 1차 배치 실행
    results = chain.batch(inputs, config={"max_concurrency": BATCH_SIZE})
    
    # 유효성 검사 및 재처리 대상 수집
    final_results = []
    error_indices = []
    error_contexts = []
    
    for i, result in enumerate(results):
        result = result.strip()
        if is_valid_context(result) and len(result) <= MAX_CHARS:
            final_results.append(result)
        else:
            final_results.append(None)  # placeholder
            error_indices.append(i)
            error_contexts.append(result)
    
    # 2차 배치: 실패한 것들 재처리 (한번에!)
    if error_contexts:
        print(f"  ⚠️ {len(error_contexts)}개 재처리 중...")
        error_inputs = [{"error_context": ctx} for ctx in error_contexts]
        fixed_results = parse_chain.batch(error_inputs, config={"max_concurrency": len(error_inputs)})
        
        for idx, fixed in zip(error_indices, fixed_results):
            final_results[idx] = fixed.strip()
    
    return final_results

print("✅ 배치 처리 함수 정의 완료")

In [ ]:
# =============================================================================
# 📦 CELL 11: 체크포인트 & 저장 함수
# =============================================================================
def save_checkpoint(documents: list, processed_videos: set, youtuber: str):
    """체크포인트 저장"""
    docs_path = f"{CHECKPOINT_PATH}/{youtuber}_checkpoint_documents.jsonl"
    videos_path = f"{CHECKPOINT_PATH}/{youtuber}_checkpoint_processed_videos.json"
    
    with open(docs_path, "w", encoding="utf-8") as f:
        for doc in documents:
            f.write(json.dumps(doc, ensure_ascii=False) + "\n")
    
    with open(videos_path, "w", encoding="utf-8") as f:
        json.dump(list(processed_videos), f, ensure_ascii=False)
    
    print(f"💾 체크포인트: {len(documents)}개 문서, {len(processed_videos)}개 영상")


def save_final(documents: list, youtuber: str):
    """최종 결과 저장"""
    output_file = f"{OUTPUT_PATH}/{youtuber}_contextualized_documents.jsonl"
    with open(output_file, "w", encoding="utf-8") as f:
        for doc in documents:
            f.write(json.dumps(doc, ensure_ascii=False) + "\n")
    print(f"✅ 최종 저장: {output_file}")

print("✅ 저장 함수 정의 완료")

In [ ]:
# =============================================================================
# 📦 CELL 12: 메인 처리 함수 (⚡ 최적화 버전)
# =============================================================================
def process_all_videos_optimized(youtuber: str, checkpoint_interval: int = 5):
    """
    전체 청크를 모아서 대형 배치로 처리
    - A100의 병렬 처리 능력 최대 활용
    - 영상별이 아닌 전체 청크 단위 배치
    """
    transcript_dir = f"{DATA_PATH}/{youtuber}/transcripts"
    meta_dir = f"{DATA_PATH}/{youtuber}/meta"
    
    transcript_paths = sorted(glob.glob(f"{transcript_dir}/*.jsonl"))
    print(f"🔍 전체 자막 파일: {len(transcript_paths)}개")
    
    # 1. 처리 대상 영상 수집
    valid_videos = []
    for file_path in tqdm(transcript_paths, desc="영상 스캔"):
        video_id = os.path.basename(file_path).split(".")[0]
        transcript_data = read_jsonl(file_path)
        if not transcript_data:
            continue
        
        recollect_id = transcript_data.get("recollect_id", 0)
        metadata = get_matching_metadata(f"{meta_dir}/{video_id}.jsonl", recollect_id)
        if not metadata or metadata.get("is_shorts", False):
            continue
        
        valid_videos.append({
            "video_id": video_id,
            "metadata": metadata,
            "transcript_data": transcript_data,
            "recollect_id": recollect_id
        })
    
    print(f"📹 처리 대상: {len(valid_videos)}개 영상")
    
    # 2. 체크포인트 로드
    processed_videos = set()
    documents = []
    videos_path = f"{CHECKPOINT_PATH}/{youtuber}_checkpoint_processed_videos.json"
    docs_path = f"{CHECKPOINT_PATH}/{youtuber}_checkpoint_documents.jsonl"
    
    if os.path.exists(videos_path):
        with open(videos_path, "r") as f:
            processed_videos = set(json.load(f))
    if os.path.exists(docs_path):
        with open(docs_path, "r") as f:
            for line in f:
                documents.append(json.loads(line))
        print(f"📂 체크포인트: {len(documents)}개 문서, {len(processed_videos)}개 영상 처리됨")
    
    # 3. 남은 영상 필터링
    remaining = [v for v in valid_videos if v["video_id"] not in processed_videos]
    print(f"⏳ 남은 영상: {len(remaining)}개")
    
    if not remaining:
        print("✅ 모든 영상 처리 완료!")
        return documents
    
    # 4. 모든 청크 수집 (영상 경계 없이)
    all_chunks = []
    
    for video_info in tqdm(remaining, desc="청크 수집"):
        transcript = video_info["transcript_data"].get("transcript", [])
        full_transcript = "\n".join([seg.get("text", "") for seg in transcript])
        if not full_transcript:
            processed_videos.add(video_info["video_id"])
            continue
        
        title = video_info["metadata"].get("title", "")
        video_duration = video_info["metadata"].get("duration")
        chunks = create_chunks_with_overlap(transcript, video_duration=video_duration)
        
        for chunk in chunks:
            all_chunks.append({
                "video_info": video_info,
                "chunk_data": chunk,
                "title": title,
                "full_transcript": full_transcript,
                "chunk": chunk["content"]
            })
    
    print(f"📦 총 청크 수: {len(all_chunks)}개")
    print(f"⚡ 예상 배치 횟수: {len(all_chunks) // BATCH_SIZE + 1}회")
    
    # 5. 대형 배치로 처리
    batch_count = 0
    for batch_start in tqdm(range(0, len(all_chunks), BATCH_SIZE), desc="배치 처리"):
        batch = all_chunks[batch_start:batch_start + BATCH_SIZE]
        batch_count += 1
        
        # 배치 문맥 생성
        contexts = batch_generate_contexts_optimized(batch)
        
        # 결과 저장
        for item, context in zip(batch, contexts):
            video_info = item["video_info"]
            chunk_data = item["chunk_data"]
            
            doc = {
                "id": None,
                "page_content": f"문맥: {context}\n\n{chunk_data['content']}",
                "metadata": {
                    "video_id": video_info["video_id"],
                    "title": item["title"],
                    "channel_name": video_info["metadata"].get("channel_name", ""),
                    "recollect_id": video_info["recollect_id"],
                    "chunk_index": chunk_data["chunk_index"],
                    "char_count": chunk_data["char_count"],
                    "prev_overlap": chunk_data["prev_overlap"],
                    "next_overlap": chunk_data["next_overlap"],
                    "start_time": chunk_data["start_time"],
                    "end_time": chunk_data["end_time"],
                },
                "type": "Document"
            }
            documents.append(doc)
            processed_videos.add(video_info["video_id"])
        
        # 체크포인트 저장
        if batch_count % checkpoint_interval == 0:
            save_checkpoint(documents, processed_videos, youtuber)
    
    # 6. 최종 저장
    save_checkpoint(documents, processed_videos, youtuber)
    save_final(documents, youtuber)
    
    return documents

print("✅ 메인 처리 함수 정의 완료")

In [ ]:
# =============================================================================
# 📦 CELL 13: 실행
# =============================================================================
youtuber = "tzuyang"
documents = process_all_videos_optimized(youtuber, checkpoint_interval=5)
print(f"\n🎉 완료! 총 {len(documents)}개 문서")

In [ ]:
# =============================================================================
# 📦 CELL 14: GPU 모니터링 (선택사항)
# =============================================================================
!nvidia-smi